In [ ]:
!pip install -q peft
!pip install -q trl
!pip install -q sentence-transformers
!pip install -q FlagEmbedding

# SCRIPT

In [1]:
from transformers import LlamaForCausalLM, AutoTokenizer, pipeline
from peft import get_peft_model, PeftModel, LoraConfig, TaskType

from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from transformers import TrainingArguments
from datasets import Dataset, load_dataset

from FlagEmbedding import BGEM3FlagModel
from sentence_transformers import SentenceTransformer

import torch
from torch.nn import functional as F
from huggingface_hub import login

import os
from datetime import datetime
from tqdm import tqdm

In [4]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HF_TOKEN = None

WORK_DIR = '/kaggle/working/'
GENERATOR_MODEL_DIR = os.path.join(WORK_DIR, 'model/generator/')

BASE_MODEL_HF_NAME = 'meta-llama/Llama-3.2-3B-Instruct'
BASE_MODEL_DIR = os.path.join(GENERATOR_MODEL_DIR, 'base/')
ADAPTER_DIR = os.path.join(GENERATOR_MODEL_DIR, 'adapter/')

DATA_DIR = '/kaggle/input/folder'
TRAIN_DATA_FILE = os.path.join(DATA_DIR, 'train.csv')
TEST_DATA_FILE = os.path.join(DATA_DIR, 'test.csv')

FINETUNE_OUTPUT_DIR = os.path.join(WORK_DIR, 'finetune_output/')

In [5]:
def download_base_model(save_dir: str = BASE_MODEL_DIR):
    login(token = HF_TOKEN)
    model_dir = BASE_MODEL_HF_NAME

    model = LlamaForCausalLM.from_pretrained(
        model_dir, 
        device_map="cpu",
    )
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir, 
        device_map="cpu",
    )
  
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    
    del model, tokenizer

In [8]:
class PairSimilarityCalc():
    def __init__(self):
        self.m3 = BGEM3FlagModel('BAAI/bge-m3')
        self.vi = SentenceTransformer('dangvantuan/vietnamese-embedding-LongContext', trust_remote_code=True)

    def calc_similarity(self, sentence_1: str, sentence_2:str):
        pair = [sentence_1, sentence_2]
        score_m3 = self.m3.compute_score(pair, max_passage_length=256, weights_for_different_modes=[1.0, 0.2, 1.0])
        
        embed_vi = self.vi.encode(pair, show_progress_bar=False)
        score_vi = F.cosine_similarity(torch.tensor(embed_vi[0]), torch.tensor(embed_vi[1]), dim=0).item()
        
        return (score_m3['dense'], score_m3['colbert+sparse+dense'], score_vi)
        

class Evaluator():
    def __init__(self, test_data_file: str):
        self.test_data_file = test_data_file
        self.dataset = self.get_dataset(test_data_file)
        self.sim_calculator = PairSimilarityCalc()
        
    
    def evaluate(self, answers) -> dict:
        ds = Dataset.from_dict({
                # "question": self.dataset['test']['Question'],
                "answer": answers,
                # "contexts": self.dataset['test']['Context'],
                "ground_truth": self.dataset['test']['Answer'],
        })
        
        # log = []
        acc_score = 0
        for sample in tqdm(ds, desc='Calculating similarity'):
            sample_score = self.sim_calculator.calc_similarity(sample['answer'], sample['ground_truth'])
            avg = (min(sample_score[0], sample_score[1]) + sample_score[2]) / 2
            # log.append(dict(...))
            acc_score += avg
        
        return acc_score / len(ds)
            
    
    def get_inference_answers(self, generator_pipeline):
        answers = []
        for chat in tqdm(self.dataset['test']['text'], desc='Inferencing'):
            answers.append(generator_pipeline(chat)[-1]['generated_text'])
        return answers
        # return generator_pipeline(self.dataset['test']['text'])  # [[{'generated_text': ...}],]
    
    def get_dataset(self, data_file: str):
        dataset = load_dataset('csv', data_files=dict(test=data_file))
        chat_template = """Bạn được cung cấp cho một ngữ cảnh. 
Chỉ dựa vào ngữ cảnh ấy, hãy trả lời cho câu hỏi bên dưới. Tuyệt đối không sử dụng thông tin bên ngoài, không có trong ngữ cảnh.

### Ngữ cảnh:
{CONTEXT}

### Câu hỏi:
{QUESTION}

### Trả lời:
"""
        dataset = dataset.map(
            lambda x: {'text': chat_template.format(
                    CONTEXT = x['Context'], 
                    QUESTION = x['Question']),}
        )
        dataset = dataset.map(lambda x: {'Context': [x['Context']]})
#         dataset = dataset.map(lambda x: {'Answer': [x['Answer']]})
        
        return dataset

In [11]:
class FinetuneEngine():
    def __init__(
        self,
        base_model_dir: str = BASE_MODEL_DIR,
        adapter_dir: str = ADAPTER_DIR,
        init_adapter: bool = False,
        train_data_file: str = TRAIN_DATA_FILE,
        test_data_file: str = TEST_DATA_FILE,
        save_model_dir: str = FINETUNE_OUTPUT_DIR,
    ) -> None:
        
        self.base_dir = base_model_dir
        self.adapter_dir = (adapter_dir if not init_adapter else None)
        self.train_data_file = train_data_file
        self.test_data_file = test_data_file
        self.save_model_dir = save_model_dir
        
        self.base_model, self.tokenizer = self.get_base_model(self.base_dir)
        self.train_dataset = self.get_train_dataset(self.train_data_file)
        self.output_dir = None
#         self.evaluator = Evaluator(self.test_data_file)
        
    
    def train(self, adapter_dir = None, init_adapter = False):
        if init_adapter:
            self.adapter_dir = None
        elif adapter_dir is not None:
            self.adapter_dir = adapter_dir
        peft_model = self.get_peft_model(self.adapter_dir)
        
        trainer =  self.get_default_sfttrainer(peft_model)
        trainer.train()

#         print('Finetuned model')
#         finetune_score = self.evaluate()
#         print('Evaluation score: ', finetune_score)
        
    def evaluate(self, lora_path = None):
        if lora_path is not None:
            self.base_model.load_adapter(lora_path, adapter_name='lora')    
        
        gen_pipeline = pipeline(
            task="text-generation", 
            model=self.base_model, 
            tokenizer=self.tokenizer,
            # do_sample=False,  # not use temperature
            temperature = 0.1,
            # max_length=2048,
            max_new_tokens=100,
            return_full_text=False,
            repetition_penalty=1.1,
            eos_token_id=self.tokenizer.eos_token_id,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        answers = self.evaluator.get_inference_answers(gen_pipeline)
        result = self.evaluator.evaluate(answers)
        if lora_path is not None:
            self.base_model.disable_adapters()
        return result
    
    ## ----------------------------------------------------------
    # load func
    def get_base_model(self, model_dir: str):
        model = LlamaForCausalLM.from_pretrained(model_dir, device_map=DEVICE)
        tokenizer = AutoTokenizer.from_pretrained(model_dir, device_map=DEVICE)
        tokenizer.pad_token_id = tokenizer.eos_token_id
        # tokenizer.pad_token =  "<|end_of_text|>"

        return model, tokenizer

    
    def get_peft_model(self, adapter_dir: str):
        if adapter_dir is None:       
            lora_config = LoraConfig(
                r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
                target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj",],
                lora_alpha = 16,
                lora_dropout = 0.05, # [0, 0.05]
                bias = "none",
                task_type=TaskType.CAUSAL_LM,
            )
            peft_model = get_peft_model(self.base_model, lora_config)
            # self.adapter_dir = os.path.join(self.save_model_dir, 'init')
            # peft_model.save_pretrained(self.old_adapter_dir)
        else:
            # lora_config = PeftConfig.from_pretrained(adapter_dir)
            peft_model = PeftModel.from_pretrained(self.base_model, adapter_dir, is_trainable=True)
        
        return peft_model
    
    
    # map train dataset
    def get_train_dataset(self, data_file: str):
        dataset = load_dataset('csv', data_files=dict(train=data_file))
        chat_template = """Bạn được cung cấp cho một ngữ cảnh. 
Chỉ dựa vào ngữ cảnh ấy, hãy trả lời cho câu hỏi bên dưới. Tuyệt đối không sử dụng thông tin bên ngoài, không có trong ngữ cảnh.

### Ngữ cảnh:
{CONTEXT}

### Câu hỏi:
{QUESTION}

### Trả lời:
{ANSWER}"""
        dataset = dataset.map(
            lambda x: {'text': chat_template.format(
                CONTEXT = x['Context'], 
                QUESTION = x['Question'], 
                ANSWER = x['Answer']
        )})
        return dataset
    
    
    # trainer
    def get_default_sfttrainer(self, peft_model):

        response_template = "\n### Trả lời:\n"
        response_template_ids = self.tokenizer.encode(response_template, add_special_tokens=False)[1:]
        collator = DataCollatorForCompletionOnlyLM(response_template_ids, tokenizer=self.tokenizer)
        # max_seq_len = self.tokenizer.model_max_length  # =131072
        max_seq_len = 2048
        self.output_dir = os.path.join(self.save_model_dir, 'run-' + datetime.now().strftime('%m%d-%H%M%S'))
        
        trainer = SFTTrainer(
            model = peft_model,
            tokenizer = self.tokenizer,
            # peft_config = lora_config,
            train_dataset = self.train_dataset['train'],
            dataset_text_field = "text",  # to create ConstantLengthDataset here
            max_seq_length = max_seq_len,
            # formatting_func=formatting_prompts_func,
            data_collator=collator,
            packing = False,
            args = TrainingArguments(
                per_device_train_batch_size = 2,
                gradient_accumulation_steps=4,
                num_train_epochs = 1,
                learning_rate = 2e-4,
                logging_steps = 5,
                optim = "adamw_torch",
                weight_decay = 0.01,
                warmup_steps = 10,
                output_dir = self.output_dir,
                # save_strategy='steps',
                # save_steps=0.1,
                # save_total_limit=2,
                # metric_for_best_model='loss',
                # load_best_model_at_end=True,
                report_to='none',
            ),
        )
        
        return trainer

In [ ]:
login(token = HF_TOKEN)
train_data_file = '/kaggle/input/vnewsqa/train.csv'
test_data_file = '/kaggle/input/vnewsqa/test.csv'

engine = FinetuneEngine(
    base_model_dir=BASE_MODEL_HF_NAME,
    init_adapter=True,
    train_data_file = train_data_file,
    test_data_file = test_data_file,
)

In [13]:
# pre-finetune eval
engine.evaluate()

In [ ]:
# finetune
engine.train()

In [ ]:
engine.evalute('/kaggle/working/finetune_output/run-1004-070238')